## Dependecies

In [ ]:
!pip install transformers
!pip install datasets
!pip install soundfile
!pip install jiwer
!pip install evaluate
!pip install wandb

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 480.6/480.6 kB 26.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 179.3/179.3 kB 16.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.8/134.8 kB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.1/194.1 kB 17.1 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2024.10.0
    Uninstalling fsspec-2024.10.0:
      Successfully uninstalled fsspec-2024.10.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2024.10.0 requires fsspec==2024.10.0, but you have fsspec 2024.9.0 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 75.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.0/84.0 kB 6.1 MB/s eta 0:00:00


In [ ]:
import wandb
wandb.login()

wandb: Using wandb-core as the SDK backend.  Please refer to https://wandb.me/wandb-core for more information.


<IPython.core.display.Javascript object>

wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize
wandb: Paste an API key from your profile and hit enter, or press ctrl+c to quit:

 ··········


wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc


True

In [ ]:
import json
import re
import os
import torch
import torchaudio
import evaluate
from jiwer import wer
from glob import glob
import numpy as np
import soundfile as sf
from tqdm import tqdm

from dataclasses import dataclass, field
from typing import Any, Dict, List, Optional, Union

from datasets import load_dataset, Dataset, DatasetDict, Audio
from transformers import Wav2Vec2Tokenizer, Wav2Vec2ForCTC, Wav2Vec2Processor, TrainingArguments, Trainer

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


## Load wav2vec2 model, tokenizer and processor

In [ ]:
# load model and tokenizer
tokenizer = Wav2Vec2Tokenizer.from_pretrained("facebook/wav2vec2-base-960h")
model = Wav2Vec2ForCTC.from_pretrained("facebook/wav2vec2-base-960h").to("cuda")
processor = Wav2Vec2Processor.from_pretrained("facebook/wav2vec2-large-960h")

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/163 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/291 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/85.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.60k [00:00<?, ?B/s]

...

preprocessor_config.json:   0%|          | 0.00/159 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/163 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/843 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/291 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/85.0 [00:00<?, ?B/s]

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Load Dataset from Disk

In [ ]:
dataset= DatasetDict.load_from_disk("/content/drive/MyDrive/EE6366_Speech_Speaker_Recognition/VIOLIN_Audio_Data/dataset_TV")

In [ ]:
sampled_dataset = dataset

In [ ]:
# # Define the sampling fraction
# sampling_fraction = 0.1

# # Randomly sample 10% of each split
# sampled_dataset = DatasetDict({
#     "train": dataset["train"].train_test_split(test_size=sampling_fraction)["test"],
#     "validate": dataset["validate"].train_test_split(test_size=sampling_fraction)["test"],
#     "test": dataset["test"].train_test_split(test_size=sampling_fraction)["test"],
# })

# # Inspect the sampled dataset
# print(sampled_dataset)

## Preprocess
1. Downsample to 16K Hz
2. Use processor to get model input

In [ ]:
sampled_dataset = sampled_dataset.cast_column(
    "file",
    Audio(sampling_rate=16000)  # Automatically resample to 16kHz
)

def preprocess_with_datasets_audio_batch(batch):
    # Initialize lists for processed data
    input_values_list = []
    labels_list = []

    for file_info, text in zip(batch["file"], batch["text"]):
        # Access the audio data
        audio = file_info["array"]  # NumPy array of audio samples
        sampling_rate = file_info["sampling_rate"]  # Sampling rate of the audio

        # Ensure the audio is mono (if not already)
        if audio.ndim > 1:  # Stereo or multi-channel
            audio = audio.mean(axis=0)  # Average across channels

        # Process audio using the processor
        input_values = processor(audio, sampling_rate=16000).input_values[0]
        input_values_list.append(input_values)

        # Process text using the tokenizer
        labels = processor.tokenizer(text, return_tensors="np", padding=True).input_ids[0]
        labels_list.append(labels)

    # Return the processed batch
    return {"input_values": input_values_list, "labels": labels_list}

sampled_dataset = sampled_dataset.map(
    preprocess_with_datasets_audio_batch,
    batched=True,
    batch_size=1,  # Process one row at a time
    num_proc=8,  # Single-threaded to simplify debugging
    remove_columns=["file", "name", "text"]
)

Map (num_proc=8):   0%|          | 0/7983 [00:00<?, ? examples/s]

Map (num_proc=8):   0%|          | 0/1007 [00:00<?, ? examples/s]

Map (num_proc=8):   0%|          | 0/1013 [00:00<?, ? examples/s]

In [ ]:
# sampled_dataset.save_to_disk("/content/drive/MyDrive/EE6366_Speech_Speaker_Recognition/VIOLIN_Audio_Data/dataset_TV_processed")

## Bakup: Use batch mapping

In [ ]:
# def preprocess_batch(batch):
#     input_values = []
#     labels = []

#     for file, text in zip(batch["file"], batch["text"]):
#         # Process audio
#         audio, sr = torchaudio.load(file)
#         assert sr == 16000
#         input_values.append(processor(audio, sampling_rate=16000).input_values[0])

#         # Process text
#         labels.append(processor.tokenizer(text, return_tensors="np").input_ids[0])

#     return {"input_values": input_values, "labels": labels}

# sampled_dataset = sampled_dataset.map(
#     preprocess_batch,
#     batched=True,
#     batch_size=4,  # Process one row at a time
#     num_proc=1,  # Single-threaded to simplify debugging
#     remove_columns=["file", "name", "text"]
# )

## Prepare datacollector and metrics

In [ ]:

@dataclass
class DataCollatorCTCWithPadding:
    """
    Data collator that will dynamically pad the inputs received.
    Args:
        processor (:class:`~transformers.Wav2Vec2Processor`)
            The processor used for proccessing the data.
        padding (:obj:`bool`, :obj:`str` or :class:`~transformers.tokenization_utils_base.PaddingStrategy`, `optional`, defaults to :obj:`True`):
            Select a strategy to pad the returned sequences (according to the model's padding side and padding index)
            among:
            * :obj:`True` or :obj:`'longest'`: Pad to the longest sequence in the batch (or no padding if only a single
              sequence if provided).
            * :obj:`'max_length'`: Pad to a maximum length specified with the argument :obj:`max_length` or to the
              maximum acceptable input length for the model if that argument is not provided.
            * :obj:`False` or :obj:`'do_not_pad'` (default): No padding (i.e., can output a batch with sequences of
              different lengths).
        max_length (:obj:`int`, `optional`):
            Maximum length of the ``input_values`` of the returned list and optionally padding length (see above).
        max_length_labels (:obj:`int`, `optional`):
            Maximum length of the ``labels`` returned list and optionally padding length (see above).
        pad_to_multiple_of (:obj:`int`, `optional`):
            If set will pad the sequence to a multiple of the provided value.
            This is especially useful to enable the use of Tensor Cores on NVIDIA hardware with compute capability >=
            7.5 (Volta).
    """

    processor: Wav2Vec2Processor
    padding: Union[bool, str] = True
    max_length: Optional[int] = None
    max_length_labels: Optional[int] = None
    pad_to_multiple_of: Optional[int] = None
    pad_to_multiple_of_labels: Optional[int] = None

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        # split inputs and labels since they have to be of different lengths and need
        # different padding methods
        input_features = [{"input_values": feature["input_values"]} for feature in features]
        label_features = [{"input_ids": feature["labels"]} for feature in features]

        batch = self.processor.pad(
            input_features,
            padding=self.padding,
            max_length=self.max_length,
            pad_to_multiple_of=self.pad_to_multiple_of,
            return_tensors="pt",
        )
        with self.processor.as_target_processor():
            labels_batch = self.processor.pad(
                label_features,
                padding=self.padding,
                max_length=self.max_length_labels,
                pad_to_multiple_of=self.pad_to_multiple_of_labels,
                return_tensors="pt",
            )

        # replace padding with -100 to ignore loss correctly
        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)

        batch["labels"] = labels

        return batch

def compute_metrics(pred):
    pred_logits = pred.predictions
    pred_ids = np.argmax(pred_logits, axis=-1)

    pred.label_ids[pred.label_ids == -100] = processor.tokenizer.pad_token_id

    pred_str = processor.batch_decode(pred_ids)
    # we do not want to group tokens when computing the metrics
    label_str = processor.batch_decode(pred.label_ids, group_tokens=False)

    wer = wer_metric.compute(predictions=pred_str, references=label_str)

    return {"wer": wer}

data_collator = DataCollatorCTCWithPadding(processor=processor, padding=True)
wer_metric = evaluate.load("wer")

## Freeze feature extractor

In [ ]:
model.freeze_feature_extractor()

## Freeze base model
# for param in model.wav2vec2.parameters():
#     param.requires_grad = False

/usr/local/lib/python3.10/dist-packages/transformers/models/wav2vec2/modeling_wav2vec2.py:2178: FutureWarning: The method `freeze_feature_extractor` is deprecated and will be removed in Transformers v5. Please use the equivalent `freeze_feature_encoder` method instead.
  warnings.warn(


## Model Params

In [ ]:
# training_args = TrainingArguments(
#     output_dir="/content/drive/MyDrive/EE6366_Speech_Speaker_Recognition/wav2vec2-finetuned",
#     evaluation_strategy="steps",
#     per_device_train_batch_size=4,
#     gradient_accumulation_steps=4,
#     num_train_epochs=10,
#     learning_rate=3e-4,
#     warmup_steps=25,
#     save_steps=50,
#     save_total_limit=2,
#     logging_steps=50,
#     fp16=True,  # Use mixed precision for faster training
#     push_to_hub=False,
#     # report_to=None,
# )

# training_args = TrainingArguments(
#     output_dir="/content/drive/MyDrive/EE6366_Speech_Speaker_Recognition/wav2vec2-finetuned",
#     evaluation_strategy="steps",
#     per_device_train_batch_size=4,
#     gradient_accumulation_steps=2,
#     num_train_epochs=10,
#     learning_rate=3e-4,
#     warmup_steps=2,
#     save_steps=4,
#     save_total_limit=2,
#     logging_steps=4,
#     fp16=True,  # Use mixed precision for faster training
#     push_to_hub=False,
#     # report_to=None,  # allow wandb
# )

training_args = TrainingArguments(
    output_dir="/content/drive/MyDrive/EE6366_Speech_Speaker_Recognition/wav2vec2-finetuned",
    evaluation_strategy="steps",
    per_device_train_batch_size=16,  # Increase based on GPU memory
    gradient_accumulation_steps=2,  # Adjust based on effective batch size
    num_train_epochs=5,  # Fewer epochs due to larger dataset
    learning_rate=5e-5,  # Lower learning rate for stability
    warmup_steps=125,  # Increase warmup steps for larger dataset
    save_steps=100,  # Save every ~epoch
    save_total_limit=2,
    logging_steps=100,  # Log every few hundred steps
    fp16=True,  # Mixed precision for faster training
    push_to_hub=False,
    load_best_model_at_end=True,  # Automatically load the best model
    metric_for_best_model="eval_loss",  # Use validation loss to pick the best model
)

/usr/local/lib/python3.10/dist-packages/transformers/training_args.py:1568: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


In [ ]:
trainer = Trainer(
    model=model,
    data_collator=data_collator,
    args=training_args,
    compute_metrics=compute_metrics,
    train_dataset=sampled_dataset['train'],
    eval_dataset=sampled_dataset['validate'],
    tokenizer=processor.feature_extractor,
)

<ipython-input-24-d8a53228fdb2>:1: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [ ]:
trainer.train()

/usr/local/lib/python3.10/dist-packages/transformers/models/wav2vec2/processing_wav2vec2.py:174: UserWarning: `as_target_processor` is deprecated and will be removed in v5 of Transformers. You can process your labels by using the argument `text` of the regular `__call__` method (either in the same call as your audio inputs, or in a separate call.
  warnings.warn(


Step,Training Loss,Validation Loss,Wer
100,9280.610000,3517.851807,0.422510
200,7987.008700,3125.365967,0.383406
300,7489.415600,2914.863525,0.377475


/usr/local/lib/python3.10/dist-packages/transformers/models/wav2vec2/processing_wav2vec2.py:174: UserWarning: `as_target_processor` is deprecated and will be removed in v5 of Transformers. You can process your labels by using the argument `text` of the regular `__call__` method (either in the same call as your audio inputs, or in a separate call.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/transformers/models/wav2vec2/processing_wav2vec2.py:174: UserWarning: `as_target_processor` is deprecated and will be removed in v5 of Transformers. You can process your labels by using the argument `text` of the regular `__call__` method (either in the same call as your audio inputs, or in a separate call.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/transformers/models/wav2vec2/processing_wav2vec2.py:174: UserWarning: `as_target_processor` is deprecated and will be removed in v5 of Transformers. You can process your labels by using the argument `text` of the regular `__call

Step,Training Loss,Validation Loss,Wer
100,9280.610000,3517.851807,0.422510
200,7987.008700,3125.365967,0.383406
300,7489.415600,2914.863525,0.377475
400,6947.836300,2784.852783,0.366204
500,6700.137500,2690.176270,0.361269
600,6631.506900,2617.609863,0.357627
700,6324.562500,2603.134521,0.351179
800,6304.869400,2612.833008,0.345433
900,6241.645000,2540.218018,0.346922
1000,6233.677500,2552.641357,0.342135


/usr/local/lib/python3.10/dist-packages/transformers/models/wav2vec2/processing_wav2vec2.py:174: UserWarning: `as_target_processor` is deprecated and will be removed in v5 of Transformers. You can process your labels by using the argument `text` of the regular `__call__` method (either in the same call as your audio inputs, or in a separate call.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/transformers/models/wav2vec2/processing_wav2vec2.py:174: UserWarning: `as_target_processor` is deprecated and will be removed in v5 of Transformers. You can process your labels by using the argument `text` of the regular `__call__` method (either in the same call as your audio inputs, or in a separate call.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/transformers/models/wav2vec2/processing_wav2vec2.py:174: UserWarning: `as_target_processor` is deprecated and will be removed in v5 of Transformers. You can process your labels by using the argument `text` of the regular `__call

TrainOutput(global_step=1245, training_loss=6838.160240963855, metrics={'train_runtime': 12491.2677, 'train_samples_per_second': 3.195, 'train_steps_per_second': 0.1, 'total_flos': 1.3333678235927654e+19, 'train_loss': 6838.160240963855, 'epoch': 4.98997995991984})

In [ ]:
print("Current Device:", torch.cuda.current_device())
print("Is Model on GPU?", next(model.parameters()).is_cuda)

Current Device: 0
Is Model on GPU? True


In [ ]:
results = trainer.evaluate()

## Evaluations

In [ ]:
def speech_to_text(speech):
  input_values = tokenizer(speech, return_tensors="pt", padding="longest").input_values
  with torch.no_grad():
      logits = model(input_values.to("cuda")).logits
      # logits = model(input_values).logits

  predicted_ids = torch.argmax(logits, dim=-1)
  transcription = tokenizer.batch_decode(predicted_ids)
  return transcription

def load_audio(file_path):
  # data_path = audio_dir+name+".flac"

  speech, _ = sf.read(file_path)
  # Check if the audio is stereo
  if len(speech.shape) == 2:  # Stereo: (num_samples, num_channels)
      # Convert to mono by averaging the two channels
      speech_mono = speech.mean(axis=1)
  else:  # Already mono: (num_samples,)
      speech_mono = speech
  return speech_mono

## The orignal problem was caused by SR.
We can run some simple evaluation using original model, by down-sampling and change channel to mono

In [ ]:
sample = "/content/drive/MyDrive/EE6366_Speech_Speaker_Recognition/VIOLIN_Audio_Data/audio_clips/friends_s05e20_clip_697_724.flac"

In [ ]:
os.path.exists(sample)

True

In [ ]:
speech, sr = downsample_audio(sample)
text = speech_to_text(speech)

In [ ]:
len(speech)

441760

In [ ]:
text

["EROS IT'S YOU I JUST WONT YOU REMEMBER THIS FEELING YOU ARE LUCKY TO BE ALIVE SO LIVE EVERY DAY TO THE FULLEST LOVE YOURSELF FO QUET I  AM ALSO GET STAMPS BUT OW SEE THAT MESSAGE FOR EMILY AND HIS WHOLE PROBINTOS REINY WEN YOU WANT IMMETING YOUR O JON"]

In [ ]:
trans = "HEY ROSS IT IS YOU I JUST WANT YOU TO REMEMBER THIS FEELING YOU ARE LUCKY TO BE ALIVE SO LIVE EVERY DAY TO THE FULLEST LOVE YOURSELF OKAY AND ALSO GET STAMPS BYE PLAY THAT MESSAGE FOR EMILY AND THIS PROBLEM GOES AWAY RIGHT ANYBODY WANT TO MEET A HERO JOHN GLENN IS HERE"

In [ ]:
trans

'HEY ROSS IT IS YOU I JUST WANT YOU TO REMEMBER THIS FEELING YOU ARE LUCKY TO BE ALIVE SO LIVE EVERY DAY TO THE FULLEST LOVE YOURSELF OKAY AND ALSO GET STAMPS BYE PLAY THAT MESSAGE FOR EMILY AND THIS PROBLEM GOES AWAY RIGHT ANYBODY WANT TO MEET A HERO JOHN GLENN IS HERE'

In [ ]:
wer(text[0], "asd asd")

1.0